In [16]:
# =========================
# Task 3: Event Impact Modeling 
# =========================

import pandas as pd
import numpy as np
import statsmodels.api as sm
from pathlib import Path

# -------------------------
# Load data safely
# -------------------------
DATA_PATH = Path("../data/processed/ethiopia_fi_enriched.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"File not found: {DATA_PATH}. Run Task 1 first.")

df = pd.read_csv(DATA_PATH)
df["observation_date"] = pd.to_datetime(df["observation_date"], errors="coerce")

# -------------------------
# Separate observations and events
# -------------------------
obs = df[df["record_type"] == "observation"].copy()
events_df = df[df["record_type"] == "event"].copy()

# -------------------------
# Prepare ACCESS: Account Ownership
# -------------------------
access = obs[
    (obs["indicator_code"] == "ACC_OWNERSHIP") &
    (obs["gender"] == "all")
].copy()

access["year"] = access["observation_date"].dt.year
access = access.dropna(subset=["year", "value_numeric"]).sort_values("year")

# -------------------------
# Prepare USAGE: Mobile Money Activity Rate (best % proxy)
# -------------------------
usage = obs[
    obs["indicator_code"].isin(["USG_ACTIVE_RATE"])
].copy()

usage["year"] = usage["observation_date"].dt.year
usage = usage.dropna(subset=["year", "value_numeric"]).sort_values("year")

# -------------------------
# Extract Key Event Years
# -------------------------
event_map = {
    "telebirr": "EVT_TELEBIRR",
    "mpesa": "EVT_MPESA",
    "fayda": "EVT_FAYDA",
    "fx_reform": "EVT_FX_REFORM",
    "nfis2": "EVT_NFIS2"
}

event_years = {}
for name, code in event_map.items():
    row = events_df[events_df["indicator_code"] == code]
    if len(row) > 0:
        event_years[name] = pd.to_datetime(row["observation_date"]).dt.year.iloc[0]

print("Detected Event Years:", event_years)

# -------------------------
# Create Event Dummies
# -------------------------
for name, year in event_years.items():
    access[name] = (access["year"] >= year).astype(int)
    usage[name] = (usage["year"] >= year).astype(int)

# -------------------------
# Event Impact Model – Access
# -------------------------
X_acc = sm.add_constant(access[["year"] + list(event_years.keys())])
y_acc = access["value_numeric"]

access_model = sm.OLS(y_acc, X_acc).fit()

print("\n=== Task 3: Event Impact – Account Ownership ===")
print(access_model.summary())

# -------------------------
# Event Impact Model – Usage
# -------------------------
if len(usage) >= 3:
    X_use = sm.add_constant(usage[["year"] + list(event_years.keys())])
    y_use = usage["value_numeric"]

    usage_model = sm.OLS(y_use, X_use).fit()

    print("\n=== Task 3: Event Impact – Mobile Money Usage ===")
    print(usage_model.summary())
else:
    usage_model = None
    print("⚠ Not enough usage data points for regression.")


Detected Event Years: {'telebirr': np.int32(2021), 'mpesa': np.int32(2023), 'fayda': np.int32(2024), 'fx_reform': np.int32(2024), 'nfis2': np.int32(2021)}

=== Task 3: Event Impact – Account Ownership ===
                            OLS Regression Results                            
Dep. Variable:          value_numeric   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                    nan
Method:                 Least Squares   F-statistic:                       nan
Date:                Wed, 04 Feb 2026   Prob (F-statistic):                nan
Time:                        23:59:52   Log-Likelihood:                 75.134
No. Observations:                   4   AIC:                            -142.3
Df Residuals:                       0   BIC:                            -144.7
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
     

c:\Users\HP\Downloads\Mercy's\Mih\Kiam\week 10\ethiopia-fi-forecast\venv\Lib\site-packages\statsmodels\stats\stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 4 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "
c:\Users\HP\Downloads\Mercy's\Mih\Kiam\week 10\ethiopia-fi-forecast\venv\Lib\site-packages\statsmodels\regression\linear_model.py:1795: RuntimeWarning: divide by zero encountered in divide
  return 1 - (np.divide(self.nobs - self.k_constant, self.df_resid)
c:\Users\HP\Downloads\Mercy's\Mih\Kiam\week 10\ethiopia-fi-forecast\venv\Lib\site-packages\statsmodels\regression\linear_model.py:1795: RuntimeWarning: invalid value encountered in scalar multiply
  return 1 - (np.divide(self.nobs - self.k_constant, self.df_resid)
c:\Users\HP\Downloads\Mercy's\Mih\Kiam\week 10\ethiopia-fi-forecast\venv\Lib\site-packages\statsmodels\regression\linear_model.py:1717: RuntimeWarning: divide by zero encountered in scala